# Cloud motion vector

Find the cloud motion vector for the AT3D-MISR-CAMP2Ex case study retrieval.

In [1]:
# conda env 'misr'
import MisrToolkit as mtk
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.MISR import hdf2xr, grp2xr

In [ ]:
## Set data path
misr_data_dir = PROJECT_ROOT / 'data' / 'raw' / 'MISR'

## Specify MISR parameters
path = 117
orbit = 105254
cameras = ['DF', 'CF', 'BF', 'AF', 'AN', 'AA', 'BA', 'CA', 'DA']

## Define geographic region
center_lat = 19.266267063099104
center_lon = 119.72428415176724
region = mtk.MtkRegion(center_lat, center_lon, 8.8*3, 8.8*3, "km")

# Instrumet Heading from new (not yet public as of 7/29/2026) TC_CLOUD product
tc_cloud_file = f'/data/gdi/e/MISR/from_asdc/Reprocessing/v2026/MIL2TCSP.001/2019.10.02/MISR_AM1_TC_CLOUD_P{path:03d}_O{orbit:06d}_F01_2026.hdf'
instrumentHeading = hdf2xr(tc_cloud_file, 'Motion_17.6_km', ['InstrumentHeading'], region)

# u and v from old TC_CLOUD product
tc_cloud_file = misr_data_dir / f'MISR_AM1_TC_CLOUD_P{path:03d}_O{orbit:06d}_F01_0001.hdf'
cmv = hdf2xr(tc_cloud_file, 'Motion_17.6_km', ['CloudTopHeightOfMotion', 'CloudMotionNorthward', 'CloudMotionEastward', 'MotionQualityIndicator'], region)


In [4]:
def rotate(u,v,ang):
    rot_u = u*np.cos(ang) - v*np.sin(ang)
    rot_v = u*np.sin(ang) + v*np.cos(ang)
    return (rot_u, rot_v)

In [5]:
heading = instrumentHeading.InstrumentHeading.data[1, 1]
# u = 1.3 
# v = 2.4
u = cmv.CloudMotionEastward.data[1,1]
v = cmv.CloudMotionNorthward.data[1,1]
angle = (90 - heading)*np.pi/180
(along, cross) = rotate(u, v, -angle)

along_sigma = 2.8
cross_sigma = 1.6
ap = rotate(along+along_sigma, cross, angle)
am = rotate(along-along_sigma, cross, angle)
cp = rotate(along, cross+cross_sigma, angle)
cm = rotate(along, cross-cross_sigma, angle)
print(f'Default: u = {u:.1f} m/s, v = {v:.1f} m/s')
print(f'Along-track + {along_sigma:.1f} m/s: u = {ap[0]:.1f} m/s, v = {ap[1]:.1f} m/s')
print(f'Along-track - {along_sigma:.1f} m/s: u = {am[0]:.1f} m/s, v = {am[1]:.1f} m/s')
print(f'Cross-track + {cross_sigma:.1f} m/s: u = {cp[0]:.1f} m/s, v = {cp[1]:.1f} m/s')
print(f'Cross-track - {cross_sigma:.1f} m/s: u = {cm[0]:.1f} m/s, v = {cm[1]:.1f} m/s')

Default: u = 1.4 m/s, v = 1.9 m/s
Along-track + 2.8 m/s: u = 0.8 m/s, v = -0.8 m/s
Along-track - 2.8 m/s: u = 1.9 m/s, v = 4.7 m/s
Cross-track + 1.6 m/s: u = 2.9 m/s, v = 1.6 m/s
Cross-track - 1.6 m/s: u = -0.2 m/s, v = 2.3 m/s


In [6]:
cmv

<xarray.Dataset>
Dimensions:                 (som_x: 3, som_y: 3)
Coordinates:
  * som_x                   (som_x) float64 1.796e+07 1.798e+07 1.799e+07
  * som_y                   (som_y) float64 2.018e+05 2.194e+05 2.370e+05
Data variables:
    lat                     (som_x, som_y) float64 19.44 19.42 ... 19.11 19.09
    lon                     (som_x, som_y) float64 119.6 119.7 ... 119.7 119.9
    gamma                   (som_x, som_y) float64 1.654 1.655 ... 1.655 1.656
    CloudTopHeightOfMotion  (som_x, som_y) float32 521.1 531.3 ... 591.1 513.6
    CloudMotionNorthward    (som_x, som_y) float32 1.202 1.466 ... 1.004 0.09489
    CloudMotionEastward     (som_x, som_y) float32 1.594 1.463 ... -0.07763
    MotionQualityIndicator  (som_x, som_y) int8 86 78 79 91 92 82 93 83 84
Attributes:
    path:         117
    orbit:        105254
    block_range:  (75, 75)
    resolution:   17600
    regionID:     P117_R17600_X17959150_17994350_Y0201850_0237050